# 00 - Configuración Inicial de Unity Catalog, Volumes y Gobierno (ElectroCasa)
Ejecutar este notebook antes de desplegar el bundle en cada entorno (`electrocasa_dev` / `electrocasa_prod`).

In [ ]:
dbutils.widgets.text("catalogo", "electrocasa_dev", "Nombre del Catálogo")
catalogo = dbutils.widgets.get("catalogo")
spark.sql(f"SET var.catalogo = {catalogo}")
print(f"Configurando catálogo: {catalogo}")

In [ ]:
%sql
CREATE CATALOG IF NOT EXISTS ${var.catalogo}
COMMENT 'Plataforma de datos Medallion para ElectroCasa';

USE CATALOG ${var.catalogo};

CREATE SCHEMA IF NOT EXISTS bronze COMMENT 'Capa de ingesta raw (Streaming Tables Auto Loader + Lakeflow Connect)';
CREATE SCHEMA IF NOT EXISTS silver COMMENT 'Capa limpia, estandarizada, deduplicada, cuarentena y SCD2';
CREATE SCHEMA IF NOT EXISTS gold   COMMENT 'Capa analítica para preguntas de negocio';

CREATE VOLUME IF NOT EXISTS default.vol_landing
COMMENT 'Almacenamiento de aterrizaje por subcarpetas para fuentes CSV y JSON';

In [ ]:
fuentes = ["ventas", "catalogo", "empleados", "resenas", "devoluciones", "sucursales"]
base_path = f"/Volumes/{catalogo}/default/vol_landing"

for fuente in fuentes:
    dbutils.fs.mkdirs(f"{base_path}/{fuente}")
    print(f"Directorio verificado: {base_path}/{fuente}")

In [ ]:
%sql
-- Permisos para Ingeniería
GRANT USE CATALOG ON CATALOG ${var.catalogo} TO `electrocasa_ingenieria`;
GRANT ALL PRIVILEGES ON SCHEMA ${var.catalogo}.default TO `electrocasa_ingenieria`;
GRANT ALL PRIVILEGES ON SCHEMA ${var.catalogo}.bronze TO `electrocasa_ingenieria`;
GRANT ALL PRIVILEGES ON SCHEMA ${var.catalogo}.silver TO `electrocasa_ingenieria`;
GRANT ALL PRIVILEGES ON SCHEMA ${var.catalogo}.gold TO `electrocasa_ingenieria`;

-- Permisos para Analistas
GRANT USE CATALOG ON CATALOG ${var.catalogo} TO `electrocasa_analistas`;
GRANT USE SCHEMA ON SCHEMA ${var.catalogo}.gold TO `electrocasa_analistas`;
GRANT SELECT ON SCHEMA ${var.catalogo}.gold TO `electrocasa_analistas`;

-- Permisos para Auditoría
GRANT USE CATALOG ON CATALOG ${var.catalogo} TO `electrocasa_auditoria`;
GRANT USE SCHEMA ON SCHEMA ${var.catalogo}.gold TO `electrocasa_auditoria`;
GRANT SELECT ON SCHEMA ${var.catalogo}.gold TO `electrocasa_auditoria`;

In [ ]:
%sql
-- Permisos de sistema para Auditoría (en Free Edition capturar error y usar Lineage de Catalog Explorer)
GRANT USE CATALOG ON CATALOG system TO `electrocasa_auditoria`;
GRANT USE SCHEMA ON SCHEMA system.access TO `electrocasa_auditoria`;
GRANT SELECT ON TABLE system.access.audit TO `electrocasa_auditoria`;
GRANT SELECT ON TABLE system.access.table_lineage TO `electrocasa_auditoria`;
GRANT SELECT ON TABLE system.access.column_lineage TO `electrocasa_auditoria`;

In [ ]:
%sql
-- Demostración de GRANT y REVOKE
GRANT SELECT ON SCHEMA ${var.catalogo}.silver TO `electrocasa_analistas`;
SHOW GRANTS ON SCHEMA ${var.catalogo}.silver;

REVOKE SELECT ON SCHEMA ${var.catalogo}.silver FROM `electrocasa_analistas`;
SHOW GRANTS ON SCHEMA ${var.catalogo}.silver;

In [ ]:
%sql
-- Función de máscara dinámica para dni y salario
CREATE OR REPLACE FUNCTION ${var.catalogo}.silver.mask_dato_sensible(valor STRING)
RETURNS STRING
RETURN CASE 
    WHEN is_account_group_member('electrocasa_ingenieria') THEN valor
    ELSE '***'
END;